# 🚿 THE LEAKY FAUCET

Alright. Alright, sit down, get comfortable, get your notebook-running-anxiety in check, because we've got a problem.

Somebody trained a plain, no-frills `LinearRegression()` on the House Prices dataset. No XGBoost, no neural net, no fifteen-layer stacked ensemble held together with duct tape and hope. And... it scored like it had seen the answer key. Because, spoiler alert without actually spoiling anything: **it had.**

**Expected / observed symptom:** run this whole notebook top to bottom and watch the final validation cell. An honest linear regression baseline on this feature set should land somewhere around **R² ≈ 0.80–0.90**. If you see a number sitting up in the 0.95+ neighborhood with suspiciously chill RMSE, congratulations, you've met the faucet. It's dripping somewhere in here. Your job is to find where.

There is **exactly one** deliberate bug in this notebook. Everything else is boring, competent, professional preprocessing... the kind of code that's never going to get a promotion but also never gets fired.

I'm not going to tell you where the leak is. That would defeat the entire point of you becoming a better engineer than the person who wrote this. What I *will* do (because I'm generous, not because I'm nice, there's a difference) is give you the math you need to recognize it once you're staring right at it.

Let's get into it.

## Setup

Boring but necessary libraries... bla bla bla

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

DATA_PATH = "dataset/train.csv"


## Load the data

Just reading a CSV. I promise nothing dramatic is happening yet. That comes later.

In [2]:
def load_data(path: str) -> pd.DataFrame:
    """Loads the raw training data from disk."""
    df = pd.read_csv(path)
    return df

df_raw = load_data(DATA_PATH)
df_raw.shape


(1460, 81)

## Clean the data

Missing values, gone. Mostly-empty columns, gone. Nothing here that would make a statistician even blink. This is the "eat your vegetables" section of the notebook 

In [3]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Handles missing values and drops columns that are mostly empty."""
    df = df.copy()

    # Drop columns with a very high proportion of missing values.
    mostly_missing = [
        col for col in df.columns
        if df[col].isna().mean() > 0.4
    ]
    df = df.drop(columns=mostly_missing)

    # Impute numeric columns with the median.
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    # Impute categorical columns with the mode.
    categorical_cols = df.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode().iloc[0])

    return df

df_clean = clean_data(df_raw)
df_clean.isna().sum().sum()


/tmp/ipykernel_568/2439527970.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


np.int64(0)

## Encode the categoricals

Turning strings into numbers so the model stops complaining. Riveting stuff. Nobody has ever gotten a bug report about `LabelEncoder` and cried tears of betrayal. That's coming later, for a *different* reason.

In [4]:
def encode_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """Label-encodes remaining categorical columns for modeling."""
    df = df.copy()
    categorical_cols = df.select_dtypes(include=["object"]).columns
    encoder = LabelEncoder()
    for col in categorical_cols:
        df[col] = encoder.fit_transform(df[col].astype(str))
    return df

df_encoded = encode_categoricals(df_clean)
df_encoded.dtypes.value_counts()


/tmp/ipykernel_568/1322204753.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


int64      72
float64     3
Name: count, dtype: int64

## Feature engineering

Okay. *Okay.* This is the fun part, and also ... I'm just going to say it once and move on ... this is the part where you should be reading every single line like it owes you money.

Feature engineering is where domain knowledge gets turned into numbers a model can chew on. Total square footage, house age, years since remodel, total bathrooms.. all completely legitimate, all things you'd actually know about a house *before* it sells. That's the whole test, remember: could you know this before you know the sale price? If yes, fair game. If no... well. Keep that question in your back pocket for the next cell.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Adds a handful of derived features that summarize raw columns."""
    df = df.copy()

    # Combine basement, first floor, and second floor square footage.
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]

    # Age of the house at time of sale.
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    # Years since the last remodel.
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

    # Combined bathroom count (full + half, including basement).
    df["TotalBath"] = (
        df["FullBath"]
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"]
        + 0.5 * df["BsmtHalfBath"]
    )

    return df

df_features = engineer_features(df_encoded)



,ValueDensity
0,121.929825
1,143.819334
2,125.139978
3,81.537566
4,113.739763


## Prep the model matrix

Split into `X` and `y`. Standard stuff. Nothing to see here, officer.

In [6]:
def prep_model_data(df: pd.DataFrame):
    """Splits the frame into feature matrix X and target vector y."""
    target = df["SalePrice"]
    features = df.drop(columns=["SalePrice", "Id"])
    return features, target

X, y = prep_model_data(df_features)
X.shape, y.shape


((1460, 78), (1460,))

## Train and evaluate

This is it. This is the moment of truth. Run this cell, look at the R², and ask yourself: does an off-the-shelf linear regression, with zero tuning, genuinely deserve this score? If your gut says "that seems too easy," trust your gut. Your gut is smarter than this notebook.

In [7]:
def train_and_evaluate(X: pd.DataFrame, y: pd.Series):
    """Fits a linear regression baseline and reports validation metrics."""
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    r2 = r2_score(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    print(f"Validation R^2: {r2:.4f}")
    print(f"Validation RMSE: ${rmse:,.2f}")

    return model, r2, rmse

model, r2, rmse = train_and_evaluate(X, y)


Validation R^2: 0.9513
Validation RMSE: $19,331.96


## 🔎 The Math Clue

Full lesson, no hand-waving, because I want you to actually *get* this and not just pattern-match your way to a fix.

Hint 1: Your model's entire job is to learn a function `f` such that `f(X) ≈ y`, where `X` is information you'd genuinely have *before* you know the answer, and `y` is `SalePrice`. That's the whole social contract of supervised learning: every column in `X` has to be knowable prior to `y` existing. Violate that, and every metric downstream ... R², RMSE, your cross-validation folds, all of it... becomes theater.

Hint 2: I wonder why it's called leaky faucet


Math Trivia Random:

```
R² = 1 − (SS_res / SS_tot) = 1 − Σ(yᵢ − ŷᵢ)² / Σ(yᵢ − ȳ)²
```

## Your Mission

Go back through this notebook, find the line where the target got laundered into a feature wearing a respectable-sounding alias, rip it out, and rerun the training cell. Confirm the R² comes back down to earth. Do not skip re-running... I will know if you just eyeballed the code and called it a day.


## Submission

Fork this repo → fix the bug → open a PR → wait for a human to confirm the fix is correct and the symptom is resolved → receive your title.


### 🏅 On confirmed fix: **Senior Plumber of Predictive Integrity, First Class**

*"You found the drip. Do you understand what you found? You found the drip! Most people run this, see a 0.95, and go frame it and hang it on the wall. You went and turned off the water main like a professional. I'm... okay, I'm a little emotional. Wear the title with pride, and for the love of variance, check your feature engineering for stowaways from now on."*